In [46]:
#importamos las librerias necesarias para los ejercicios de redes neuronales convolucionales
import tensorflow as tf
import kagglehub
import glob
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import PIL

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from PIL import Image
from glob import glob
from tensorflow.keras.applications.resnet import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam


In [ ]:
# Descarga de la ultiama version del dataset de Kaggle
path = kagglehub.dataset_download("chrisfilo/fruit-recognition")
print("Path to dataset files:", path)

#Creacion de clases para generar el dataset
classes = sorted([d for d in os.listdir(path)
                  if os.path.isdir(os.path.join(path, d))])

#Creacion de la lista de rutas de imagenes
image_paths = []
for ext in ("jpg", "jpeg", "png", "JPG", "JPEG", "PNG"):
    image_paths.extend(glob(os.path.join(path, "*", f"*.{ext}")))

#Creacion de un DataFrame con las rutas de imagenes y sus etiquetas
df = pd.DataFrame({'image_path': image_paths})
df['label'] = df['image_path'].apply(lambda p: os.path.basename(os.path.dirname(p)))
print("Se ha creeado el dataframe con la informacion de las imagenes y sus etiquetas")


#Generamos los datos para las etiquetas 
le = LabelEncoder()
df["label_id"] = le.fit_transform(df["label"])

#Particionamos el dataset en entrenamiento y prueba
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["label_id"], random_state=12345,)


Path to dataset files: C:\Users\josej\.cache\kagglehub\datasets\chrisfilo\fruit-recognition\versions\1
Se ha creeado el dataframe con la informacion de las imagenes y sus etiquetas
Se ha creeado el dataframe con la informacion de las imagenes y sus etiquetas


In [48]:

model = ResNet50(input_shape=None,
                 classes=1000,
                 include_top=True,
                 weights='imagenet')


In [3]:
backbone = ResNet50(
    input_shape=(150, 150, 3), 
    weights='imagenet', 
    include_top=False)

# congela ResNet50 sin la parte superior para evitar sobreentrenamiento y mantener los pesos preentrenados
backbone.trainable = False

model = Sequential()
model.add(backbone)
model.add(GlobalAveragePooling2D())
model.add(Dense(12, activation='softmax'))

In [51]:
def load_data(df, width, height, batch_size, directory=None):
    # cuando crees tu generador de datos, agrega reescalamiento, así como volteo horizontal y vertical.
        # revisa la documentación de Keras para más técnicas
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        horizontal_flip=True,
        vertical_flip=True
    )    # especifica tu generador de datos con parámetros apropiados
        
    train_generator = train_datagen.flow_from_dataframe(
        df,
        directory=directory,
        x_col='image_path',
        y_col='label',
        target_size=(width, height),
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=True,
        )
    return train_generator
        
def create_resnet_model(input_shape):
    backbone = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    # usaremos un modelo previamente entrenado con sus parámetros aprendidos como funciones para entrenar nuestro nuevo modelo
    for layer in backbone.layers:
        layer.trainable = False
    
    # Construyamos nuestro nuevo modelo capa por capa. Comienza utilizando la capa secuencial; después 
    # agrega el backbone, una capa que aplana nuestra matriz en vectores y, por último, el 
    # encabezado de la clasificación.  No olvides ajustar el número de clases que tu modelo 
    # debe manejar  
   
    model = Sequential()# inicializa el modelo al que apilarás las capas posteriores
    model.add(backbone)# aquí va tu modelo de preentrenamiento 
    model.add(GlobalAveragePooling2D())# aquí va la capa que aplana tu modelo de salida de preentrenamiento
    model.add(Dense(12, activation='softmax'))# aquí va tu encabezado de clasificación
    
    ## Define un optimizador de tu elección. No olvides pasar un
    # parámetro learning_rate adecuado.
    
    optimizer = Adam(learning_rate=0.001)  # Aquí puedes ajustar el learning rate según tus necesidades
    
    ## compila tu modelo
    #### AQUÍ VA TU CÓDIGO ####
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model
    
def train_model(model, train_data, val_data, epochs=10):
    model.fit(
        train_data,
        validation_data=val_data,
        epochs=epochs,
        verbose=1)
    return model

In [ ]:
## Parámetros
# Dependiendo de tu máquina, puedes jugar con los parámetros de ancho y alto
# para un entrenamiento más rápido
width = 160
height = 128
batch_size = 16
input_shape = (width, height,3)

## Datos almacenados
#path_train = ""
#path_val = ""

# Carga los datos de entrenamiento y validación
train_gen = load_data(train_df, width, height , batch_size)
val_gen = load_data(test_df, width, height , batch_size)

# Crea y entrena el modelo simple
model = create_resnet_model(input_shape)
trained_model = train_model(model, train_gen, val_gen, epochs=10)

Found 49921 validated image filenames belonging to 12 classes.
Found 12481 validated image filenames belonging to 12 classes.
Epoch 1/10
3121/3121 ━━━━━━━━━━━━━━━━━━━━ 23600s 8s/step - accuracy: 0.2506 - loss: 2.2197 - val_accuracy: 0.3133 - val_loss: 2.0840
Epoch 2/10
3121/3121 ━━━━━━━━━━━━━━━━━━━━ 1604s 514ms/step - accuracy: 0.3498 - loss: 2.0132 - val_accuracy: 0.3819 - val_loss: 1.9521
Epoch 3/10
3121/3121 ━━━━━━━━━━━━━━━━━━━━ 1538s 493ms/step - accuracy: 0.3954 - loss: 1.9050 - val_accuracy: 0.3998 - val_loss: 1.8603
Epoch 4/10
3121/3121 ━━━━━━━━━━━━━━━━━━━━ 1546s 495ms/step - accuracy: 0.4219 - loss: 1.8316 - val_accuracy: 0.4398 - val_loss: 1.7969
Epoch 5/10
3121/3121 ━━━━━━━━━━━━━━━━━━━━ 1532s 491ms/step - accuracy: 0.4417 - loss: 1.7716 - val_accuracy: 0.4554 - val_loss: 1.7412
Epoch 6/10
3121/3121 ━━━━━━━━━━━━━━━━━━━━ 1408s 451ms/step - accuracy: 0.4606 - loss: 1.7254 - val_accuracy: 0.4760 - val_loss: 1.7021
Epoch 7/10
2120/3121 ━━━━━━━━━━━━━━━━━━━━ 6:24:35 23s/step - accur